# 03 — Silver Transformations

Reads from bronze VARIANT tables and builds normalized silver tables using:
- `data:field::type` extraction syntax on VARIANT columns
- MD5 surrogate keys for deduplication
- MERGE (upsert) for idempotent loads
- PK/FK constraints (RELY) for query optimization and ERD visualization
- Liquid clustering on the events table

**Tables created:**
- `venues` — PK: `venue_sk = MD5(venue_id)`
- `attractions` — PK: `attraction_sk = MD5(attraction_name || segment_name)`
- `classifications` — PK: `classification_sk = MD5(segment_name || type_name)`
- `events` — PK: `event_sk = MD5(event_name || event_datetime)`
- `event_attractions` — composite PK: `(event_sk_fk, attraction_sk_fk)`

In [ ]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "ticketmaster_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"

print(f"Bronze: {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver: {UC_CATALOG}.{SILVER_SCHEMA}")

In [ ]:
def add_pk(table: str, name: str, cols: list):
    """Add a primary key constraint if it doesn't already exist."""
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'PRIMARY KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        print(f"  PK '{name}' already exists on {table}")
        return
    for c in cols:
        try:
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {c} SET NOT NULL")
        except Exception as e:
            if "ALREADY_NOT_NULLABLE" not in str(e):
                raise
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({', '.join(cols)}) RELY")
    print(f"  Added PK '{name}' on {table}")


def add_fk(table: str, name: str, cols: list, ref_table: str, ref_cols: list):
    """Add a foreign key constraint if it doesn't already exist."""
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'FOREIGN KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        print(f"  FK '{name}' already exists on {table}")
        return
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} FOREIGN KEY ({', '.join(cols)}) REFERENCES {ref_table} ({', '.join(ref_cols)})")
    print(f"  Added FK '{name}' on {table}")

print("Helpers loaded")

## Venues

In [ ]:
silver_venues = f"{UC_CATALOG}.{SILVER_SCHEMA}.venues"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_venues} (
        venue_sk        STRING NOT NULL,
        venue_id        STRING,
        venue_name      STRING,
        venue_type      STRING,
        locale          STRING,
        postal_code     STRING,
        timezone        STRING,
        city            STRING,
        state           STRING,
        state_code      STRING,
        country         STRING,
        country_code    STRING,
        address_line1   STRING,
        longitude       DOUBLE,
        latitude        DOUBLE,
        venue_url       STRING,
        _ingestion_timestamp TIMESTAMP
    )
""")

spark.sql(f"""
    MERGE INTO {silver_venues} AS t
    USING (
        SELECT
            MD5(data:id::string) AS venue_sk,
            data:id::string               AS venue_id,
            data:name::string             AS venue_name,
            data:type::string             AS venue_type,
            data:locale::string           AS locale,
            data:postalCode::string       AS postal_code,
            data:timezone::string         AS timezone,
            data:city.name::string        AS city,
            data:state.name::string       AS state,
            data:state.stateCode::string  AS state_code,
            data:country.name::string     AS country,
            data:country.countryCode::string AS country_code,
            data:address.line1::string    AS address_line1,
            data:location.longitude::double AS longitude,
            data:location.latitude::double  AS latitude,
            data:url::string              AS venue_url,
            _ingestion_timestamp
        FROM {UC_CATALOG}.{BRONZE_SCHEMA}.venues_raw
        WHERE data:id IS NOT NULL
    ) AS s
    ON t.venue_sk = s.venue_sk
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

count = spark.table(silver_venues).count()
print(f"Venues: {count:,} rows")

## Attractions

In [ ]:
silver_attractions = f"{UC_CATALOG}.{SILVER_SCHEMA}.attractions"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_attractions} (
        attraction_sk   STRING NOT NULL,
        attraction_id   STRING,
        attraction_name STRING,
        attraction_type STRING,
        locale          STRING,
        attraction_url  STRING,
        is_test         BOOLEAN,
        segment_id      STRING,
        segment_name    STRING,
        genre_id        STRING,
        genre_name      STRING,
        _ingestion_timestamp TIMESTAMP
    )
""")

spark.sql(f"""
    MERGE INTO {silver_attractions} AS t
    USING (
        SELECT
            MD5(CONCAT_WS('||',
                COALESCE(data:name::string, ''),
                COALESCE(cl.segment_name, 'NONE')
            )) AS attraction_sk,
            data:id::string           AS attraction_id,
            data:name::string         AS attraction_name,
            data:type::string         AS attraction_type,
            data:locale::string       AS locale,
            data:url::string          AS attraction_url,
            data:test::boolean        AS is_test,
            cl.segment_id,
            cl.segment_name,
            cl.genre_id,
            cl.genre_name,
            a._ingestion_timestamp
        FROM {UC_CATALOG}.{BRONZE_SCHEMA}.attractions_raw a
        LATERAL VIEW OUTER EXPLODE(
            FROM_JSON(
                data:classifications::string,
                'ARRAY<STRUCT<segment:STRUCT<id:STRING,name:STRING>,genre:STRUCT<id:STRING,name:STRING>>>'
            )
        ) AS cl
        WHERE data:id IS NOT NULL
    ) AS s
    ON t.attraction_sk = s.attraction_sk
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

count = spark.table(silver_attractions).count()
print(f"Attractions: {count:,} rows")

## Classifications

In [ ]:
silver_classifications = f"{UC_CATALOG}.{SILVER_SCHEMA}.classifications"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_classifications} (
        classification_sk STRING NOT NULL,
        segment_id        STRING,
        segment_name      STRING,
        genre_id          STRING,
        genre_name        STRING,
        subgenre_id       STRING,
        subgenre_name     STRING,
        type_id           STRING,
        type_name         STRING,
        subtype_id        STRING,
        subtype_name      STRING,
        is_family         BOOLEAN,
        _ingestion_timestamp TIMESTAMP
    )
""")

spark.sql(f"""
    MERGE INTO {silver_classifications} AS t
    USING (
        SELECT
            MD5(CONCAT_WS('||',
                COALESCE(data:segment.name::string, 'NONE'),
                COALESCE(data:type.name::string, 'NONE')
            )) AS classification_sk,
            data:segment.id::string    AS segment_id,
            data:segment.name::string  AS segment_name,
            data:genre.id::string      AS genre_id,
            data:genre.name::string    AS genre_name,
            data:subGenre.id::string   AS subgenre_id,
            data:subGenre.name::string AS subgenre_name,
            data:type.id::string       AS type_id,
            data:type.name::string     AS type_name,
            data:subType.id::string    AS subtype_id,
            data:subType.name::string  AS subtype_name,
            data:family::boolean       AS is_family,
            _ingestion_timestamp
        FROM {UC_CATALOG}.{BRONZE_SCHEMA}.classifications_raw
    ) AS s
    ON t.classification_sk = s.classification_sk
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

count = spark.table(silver_classifications).count()
print(f"Classifications: {count:,} rows")

## Events

In [ ]:
silver_events = f"{UC_CATALOG}.{SILVER_SCHEMA}.events"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_events} (
        event_sk            STRING NOT NULL,
        event_id            STRING,
        event_name          STRING,
        event_type          STRING,
        event_url           STRING,
        locale              STRING,
        event_info          STRING,
        please_note         STRING,
        price_min           DOUBLE,
        price_max           DOUBLE,
        price_currency      STRING,
        event_date          DATE,
        event_time          STRING,
        event_datetime      TIMESTAMP,
        event_timezone      STRING,
        status_code         STRING,
        sales_start_datetime TIMESTAMP,
        sales_end_datetime  TIMESTAMP,
        is_test             BOOLEAN,
        venue_id            STRING,
        venue_sk            STRING,
        segment_name        STRING,
        type_name           STRING,
        classification_sk   STRING,
        _ingestion_timestamp TIMESTAMP
    )
    CLUSTER BY (event_date, status_code)
""")

spark.sql(f"""
    MERGE INTO {silver_events} AS t
    USING (
        SELECT
            MD5(CONCAT_WS('||',
                COALESCE(data:name::string, ''),
                COALESCE(data:dates.start.dateTime::string, '1970-01-01')
            )) AS event_sk,
            data:id::string                           AS event_id,
            data:name::string                         AS event_name,
            data:type::string                         AS event_type,
            data:url::string                          AS event_url,
            data:locale::string                       AS locale,
            data:info::string                         AS event_info,
            data:pleaseNote::string                   AS please_note,
            data:priceRanges[0].min::double           AS price_min,
            data:priceRanges[0].max::double           AS price_max,
            data:priceRanges[0].currency::string      AS price_currency,
            data:dates.start.localDate::date          AS event_date,
            data:dates.start.localTime::string        AS event_time,
            data:dates.start.dateTime::timestamp      AS event_datetime,
            data:dates.timezone::string               AS event_timezone,
            data:dates.status.code::string            AS status_code,
            data:sales.public.startDateTime::timestamp AS sales_start_datetime,
            data:sales.public.endDateTime::timestamp  AS sales_end_datetime,
            data:test::boolean                        AS is_test,
            -- Latest venue from embedded array
            data:`_embedded`.venues[0].id::string     AS venue_id,
            MD5(data:`_embedded`.venues[0].id::string) AS venue_sk,
            -- Classification
            data:classifications[0].segment.name::string AS segment_name,
            data:classifications[0].type.name::string    AS type_name,
            MD5(CONCAT_WS('||',
                COALESCE(data:classifications[0].segment.name::string, 'NONE'),
                COALESCE(data:classifications[0].type.name::string, 'NONE')
            )) AS classification_sk,
            _ingestion_timestamp
        FROM {UC_CATALOG}.{BRONZE_SCHEMA}.events_raw
        WHERE data:id IS NOT NULL
    ) AS s
    ON t.event_sk = s.event_sk
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

count = spark.table(silver_events).count()
print(f"Events: {count:,} rows")

## Event-Attractions Bridge Table

In [ ]:
silver_ea = f"{UC_CATALOG}.{SILVER_SCHEMA}.event_attractions"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_ea} (
        event_sk_fk      STRING NOT NULL,
        attraction_sk_fk STRING NOT NULL,
        event_id         STRING,
        attraction_id    STRING,
        _ingestion_timestamp TIMESTAMP
    )
""")

spark.sql(f"""
    MERGE INTO {silver_ea} AS t
    USING (
        SELECT
            MD5(CONCAT_WS('||',
                COALESCE(data:name::string, ''),
                COALESCE(data:dates.start.dateTime::string, '1970-01-01')
            )) AS event_sk_fk,
            MD5(CONCAT_WS('||',
                COALESCE(attr.name::string, ''),
                COALESCE(data:classifications[0].segment.name::string, 'NONE')
            )) AS attraction_sk_fk,
            data:id::string   AS event_id,
            attr.id::string   AS attraction_id,
            _ingestion_timestamp
        FROM {UC_CATALOG}.{BRONZE_SCHEMA}.events_raw
        LATERAL VIEW OUTER EXPLODE(
            FROM_JSON(
                data:`_embedded`.attractions::string,
                'ARRAY<STRUCT<id:STRING,name:STRING>>'
            )
        ) AS attr
        WHERE data:id IS NOT NULL AND attr.id IS NOT NULL
    ) AS s
    ON t.event_sk_fk = s.event_sk_fk AND t.attraction_sk_fk = s.attraction_sk_fk
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

count = spark.table(silver_ea).count()
print(f"Event-Attractions: {count:,} rows")

## Add PK/FK Constraints

In [ ]:
print("Adding primary key constraints...")
add_pk(f"{UC_CATALOG}.{SILVER_SCHEMA}.venues",          "venues_pk",          ["venue_sk"])
add_pk(f"{UC_CATALOG}.{SILVER_SCHEMA}.attractions",     "attractions_pk",     ["attraction_sk"])
add_pk(f"{UC_CATALOG}.{SILVER_SCHEMA}.classifications", "classifications_pk", ["classification_sk"])
add_pk(f"{UC_CATALOG}.{SILVER_SCHEMA}.events",          "events_pk",          ["event_sk"])
add_pk(f"{UC_CATALOG}.{SILVER_SCHEMA}.event_attractions", "event_attractions_pk", ["event_sk_fk", "attraction_sk_fk"])

print("\nAdding foreign key constraints...")
add_fk(f"{UC_CATALOG}.{SILVER_SCHEMA}.event_attractions", "ea_event_fk",
       ["event_sk_fk"], f"{UC_CATALOG}.{SILVER_SCHEMA}.events", ["event_sk"])
add_fk(f"{UC_CATALOG}.{SILVER_SCHEMA}.event_attractions", "ea_attraction_fk",
       ["attraction_sk_fk"], f"{UC_CATALOG}.{SILVER_SCHEMA}.attractions", ["attraction_sk"])

print("\nAll constraints applied.")

In [ ]:
# Verify: row counts and duplicate check
print("=" * 60)
print("SILVER LAYER SUMMARY")
print("=" * 60)

for table in ["venues", "attractions", "classifications", "events", "event_attractions"]:
    full = f"{UC_CATALOG}.{SILVER_SCHEMA}.{table}"
    count = spark.table(full).count()
    print(f"  {full}: {count:,} rows")

# Check for duplicate PKs
print("\nDuplicate SK check:")
for table, pk in [("venues", "venue_sk"), ("attractions", "attraction_sk"),
                  ("classifications", "classification_sk"), ("events", "event_sk")]:
    full = f"{UC_CATALOG}.{SILVER_SCHEMA}.{table}"
    dupes = spark.sql(f"SELECT {pk}, COUNT(*) c FROM {full} GROUP BY {pk} HAVING c > 1").count()
    status = "PASS" if dupes == 0 else f"FAIL ({dupes} duplicates)"
    print(f"  {table}.{pk}: {status}")

print("\nSilver layer complete.")